In [ ]:
# run only for Colab to set up environment
!pip install lisaorbits lisainstrument pytdi

In [ ]:
import numpy as np
import scipy.signal as si
import matplotlib.pyplot as plt

from scipy.constants import c

# Conventions

To understand the indexing in this notebook we first need to look at the naming conventions for LISA:
1. spacecraft (SC) are counted looking from above in clockwise order
2. each movable optical subassembly (MOSA) is identified by two indices: the index of the hosting SC and the index of the far SC
3. each measurement is indexed by the MOSA it is taken on
4. delays follow the same convention

![image.png](https://raw.githubusercontent.com/marstaa/lisa-simulation-basics/refs/heads/main/figures/indexing_scaled.png)

In [ ]:
# standard order of links/MOSAs
from lisaorbits import LINKS
print(LINKS)

# Generate orbits with LISA Orbits

LISA Orbits is a Python package that simulates the orbits of the three spacecraft in the solar system. It also computes the spacecraft velocities and light travel times between them. The results of a simulation is stored as an orbit file which can be used my the instrument simulator later on.

Available orbit types are (more and more realistic):
* `StaticConstellation` (frozen LISA at given coordinates in the solar system)
* `EqualArmlengthOrbits` (orbits optimized to leave the inter-spacecraft distances approximately constant)
* `KeplerianOrbits` (better approximation to the two-body problem)
* `OEMOrbits` (orbits including the effect of the earth-moon system and the outer planets, derived from numerical simulations done by ESA)

For more information check out the [documentation for LISA Orbits](https://lisa-simulation.pages.in2p3.fr/orbits/index.html).

In [ ]:
from lisaorbits import KeplerianOrbits, EqualArmlengthOrbits

In [ ]:
# define orbits
orbits = KeplerianOrbits()

In [ ]:
times = np.linspace(0, 3600 * 24 * 365, 1000) # in seconds

ltts = orbits.compute_ltt(times) # compute light travel times (LTTs)

print(f"array of light travel times has shape (times, links) -> {ltts.shape}")

plt.plot(times / (3600 * 24), ltts, label=[f"Link {mosa}" for mosa in LINKS])

plt.xlabel("Times (days)")
plt.ylabel("Light travel times (s)")
plt.legend()

In [ ]:
# write orbit file for the instrument simulation
orbits.write("orbits.h5", dt=3600*24, size=365, mode="w")

# Instrument simulation with LISA Instrument

LISA Instrument is a Python package that simulates the measurements of the three LISA spacecraft. Those include the so called beatnote frequencies of the three interferometers: the science interferometer (SCI), the reference interferometer (REF) and the test-mass interferometer (TMI). Together they keep track of relative distance changes between the satellites.

![image.png](https://raw.githubusercontent.com/marstaa/lisa-simulation-basics/refs/heads/main/figures/optical_bench.png)

Individually, they contain various instrumental noises

$
\begin{align}
\mathrm{sci}_{12} &= \mathbf{D}_{12} p_{21} - p_{12} + n^\mathrm{isc}_{12}, \\
\mathrm{ref}_{12} &= p_{13} - p_{12} + n^\mathrm{rfi}_{12}, \\
\mathrm{tmi}_{12} &= p_{13} - p_{12} + n^\mathrm{tmi}_{12} + 2 \delta_{12},
\end{align}
$

where $p_{ij}$ denotes laser frequency noise, $n^\mathrm{ifo}_{ij}$ optical metrology system (OMS) noise and $\delta_{ij}$ test mass acceleration noise. The delay operator $\mathbf{D}_{12}$ propagates the laser from the far to the local spacecraft. Laser frequency noise is completely dominating the measurement and has to be removed on ground by applying time-delay interferometry (TDI).

The results of a simulation is stored as an instrument file which can be imported by PyTDI to perform basic post-processing of the raw data.

For more information check out the [documentation for LISA Instrument](https://lisa-simulation.pages.in2p3.fr/instrument/).

In [ ]:
from lisainstrument import Instrument

In [ ]:
fs = 4 # sampling rate of the data (Hz)
size = (3600 * 24) * fs # one day of simulation in number of samples

instru = Instrument(
    size=size,
    dt=1 / fs,
    orbits="orbits.h5", # path to orbit file
    orbit_dataset="tcb/ltt", # choose time frame of the measurements
    lock="six", # disable laser locking, each laser is stabilized on its own cavity
    oms_asds=(6.35e-12, 0, 0, 0, 0, 0), # OMS noise level in (sci_carrier, sci_usb, tmi_carrier, tmi_usb, ref_carrier, ref_usb) in m/rtHz
    testmass_asds=2.4e-15, # TM noise level in m/s^2/rtHz
)

# only include OMS and test-mass noise in the simulation
# laser frequency noise is turned off for now to even "see" OMS and test-mass noise
instru.disable_all_noises(excluding=["oms", "test-mass"])

# run simulation and store result
instru.export_hdf5("lisa.h5", overwrite=True)

# Data processing with PyTDI

PyTDI is a Python package that performs time-delay interferometry to suppress laser frequency noise. As an input it takes the three interferometric measurements SCI, REF and TMI taken on each of the six optical benches. It then adds up (delayed) copies of those measurements to form laser noise free combinations, so called TDI combinations. The most common example is the Michelson combination 

$
\begin{align}
\xi_{12} &= \mathrm{sci}_{12} + \frac{\mathrm{ref}_{12} - \mathrm{tmi}_{12}}{2} + \mathbf{D}_{12} \frac{\mathrm{ref}_{21} - \mathrm{tmi}_{21}}{2}, \\
\eta_{12} &= \xi_{12} + \mathbf{D}_{12} \frac{\mathrm{ref}_{21} - \mathrm{ref}_{23}}{2}, \\
X_2 &= (1 - \mathbf{D}_{121} - \mathbf{D}_{12131} + \mathbf{D}_{1312121})(\eta_{13} + \mathbf{D}_{13} \eta_{31}) - (1 - \mathbf{D}_{131} - \mathbf{D}_{13121} + \mathbf{D}_{1213131})(\eta_{12} + \mathbf{D}_{12} \eta_{21}),
\end{align}
$

Here, we show the second-generation variant that suppresses laser frequency noise for a flexing LISA constellation (time-varying arms). The intermediary variables $\xi$ and $\eta$ remove spacecraft jitter and part of the laser noise, respectively. The $\eta$ variables are then the vasic building blocks for every TDI combination.

For more information check out the [documentation for PyTDI](https://lisa.pages.in2p3.fr/LDPG/wg6_inrep/pytdi/).

In [ ]:
from pytdi import Data
from pytdi.michelson import X2

In [ ]:
# Load simulated data
data = Data.from_instrument("lisa.h5")
print("available measurements: ", data.measurements.keys())

In [ ]:
# Nominal noise parameterizations

nu0 = 281600000000000.0 # central laser frequency in Hz
lambda0 = c / nu0

def asd_oms(f, level=6.35e-12, f_knee=2e-3):
    """Amplitude spectral density of OMS noise

    Args:
        level: overall level in units of m/rtHz
        f_knee: frequency below the ASD increases
    """
    level_nu = level * (2 * np.pi * f) / lambda0 # convert from displacement units to frequency units
    return level_nu * np.sqrt((1 + (f_knee / f)**4))
    
def asd_tm(f, level=2.4e-15, f_knee=4e-4, f_break=8e-3):
    """Amplitude spectral density of test mass acceleration noise

    Args:
        level: overall level in units of m/s^2/rtHz
        f_knee: frequency below the ASD increases
        f_break: frequency above the ASD increases
    """
    level_nu = level / (2 * np.pi * f) / lambda0 # convert from acceleration units to frequency units
    return level_nu * np.sqrt((1 + (f_knee / f)**2) * (1 + (f / f_break)**4))

In [ ]:
# options for spectral estimation
BUFFER = 1000 # number samples to discard at the beginning and the end of each TDI combination to remove boundary effects
welch_kwargs = dict(window=("kaiser", si.kaiser_beta(240)), nperseg=50_000, fs=fs, detrend=None)

In [ ]:
# inspect the data in the Fourier domain by estimating ASDs and comparing them to the noise models
freqs, psd_sci_12 = si.welch(data.measurements["sci_12"][BUFFER:-BUFFER], **welch_kwargs)
freqs, psd_tmi_12 = si.welch(data.measurements["tmi_12"][BUFFER:-BUFFER], **welch_kwargs)

plt.loglog(freqs, np.sqrt(psd_sci_12), label="SCI data")
plt.loglog(freqs[1:], asd_oms(freqs[1:]), label="SCI model")

plt.loglog(freqs, np.sqrt(psd_tmi_12), label="TMI data")
plt.loglog(freqs[1:], 2 * asd_tm(freqs[1:]), label="TMI model")

plt.xlabel("Frequency (Hz)")
plt.ylabel("ASD (Hz/rtHz)")
plt.legend()

In [ ]:
# build the second-generation Michelson combination and plot it in the time domain
times = np.arange(size) / fs
x2 = X2.build(**data.args)(data.measurements)

plt.plot(times / 3600, x2)

plt.xlabel("Time (h)")
plt.ylabel("TDI combination $X_2$ (Hz)")

In [ ]:
# define TDI transfer functions to model the effect of TDI on the noises contained in the interferometers
def tf_tdi_oms(f, d=8.332):
    """TDI transfer function for OMS noise"""
    omega = 2 * np.pi * f
    return np.abs(4 * np.sin(omega * d) * np.sin(2 * omega * d))

def tf_tdi_tm(f, d=8.332):
    """TDI transfer function for test-mass acceleration noise"""
    omega = 2 * np.pi * f
    return np.abs(4 * np.sin(omega * d) * np.sin(2 * omega * d)) * np.sqrt(3 + np.cos(2 * omega * d))

In [ ]:
# inspect the TDI combination in the Fourier domain by estimating its ASD and comparing it to the noise models
freqs, psd = si.welch(x2[BUFFER:-BUFFER], **welch_kwargs)

plt.loglog(freqs, np.sqrt(psd), label=r"$X_2$ data")
plt.loglog(freqs[1:], 2 * tf_tdi_oms(freqs[1:]) * asd_oms(freqs[1:]), label="OMS model")
plt.loglog(freqs[1:], 2 * tf_tdi_tm(freqs[1:]) * asd_tm(freqs[1:]), label="TM model")

plt.ylim(1e-10, 1e-3)

plt.xlabel("Frequency (Hz)")
plt.ylabel("ASD (Hz/rtHz)")
plt.legend()